# MERSCOPE ROI-Based Vascular Cell Identification and Islet Annotation Pipeline

## Identification of Islet and Exocrine Vascular Populations from Human Pancreatic MERFISH Data

This notebook processes raw Vizgen MERSCOPE spatial transcriptomic datasets to identify endothelial and perivascular cell populations within manually annotated pancreatic islets and surrounding exocrine tissue. The workflow combines ROI-based spatial annotation, quality control, dimensionality reduction, clustering, and marker-based cell type assignment to generate spatially resolved endothelial and pericyte populations for downstream extracellular matrix (ECM) and basement membrane (BM) analyses.

---

## Input Data

### Vizgen MERSCOPE Outputs

- `cell_by_gene.csv`
- `cell_metadata.csv`
- `micron_to_mosaic_pixel_transform.csv`

### ROI Annotation Files

- MERSCOPE Viewer geometry CSV files
- Polygon coordinates defining manually annotated islets

---

## Output Objects

| Object | Description |
|----------|-------------|
| `adata` | Full MERFISH dataset |
| `islet_cells` | Cells located within manually annotated islets |
| `sub_cell_by_gene` | Cells outside all islet ROIs (exocrine compartment) |
| `pericyteislet_filtered` | Islet-associated pericytes |
| `endoislet_filtered` | Islet-associated endothelial cells |

---

# PART 1 — Data Loading and Quality Control

Raw MERFISH datasets generated using the Vizgen MERSCOPE platform were imported using Squidpy's `sq.read.vizgen()` function. For datasets containing multiple imaging regions, individual AnnData objects were loaded independently and merged using `scanpy.concat()` while preserving ROI-specific cell identifiers.

Raw transcript counts were stored in:

```python
adata.layers["counts"]
```

prior to normalization and downstream analyses.

Quality control metrics were calculated using Scanpy, including:

- Total transcript counts per cell
- Number of detected genes per cell
- Transcript counts per field of view (FOV)
- Segmented cell volume
- Blank-gene abundance

Diagnostic visualizations included histograms of transcript counts, detected genes, cell volumes, and transcript counts per FOV, together with scatterplots comparing total transcripts and detected genes per cell.

Cells containing fewer than 10 transcripts were removed:

```python
sc.pp.filter_cells(adata, min_counts=10)
```

to eliminate low-quality segmented objects.

---

# PART 2 — Manual Islet Annotation

To identify endocrine regions, manually annotated islet ROIs exported from MERSCOPE Viewer were imported as geometry files.

Polygon coordinates were extracted from the geometry column and converted into coordinate arrays suitable for downstream spatial analysis. For each ROI, bounding boxes were computed to improve computational efficiency during point-in-polygon calculations.

The resulting ROI table contained:

- ROI identifier
- Polygon coordinates
- Dataset annotation
- ROI group assignment
- Bounding box coordinates

Cell centroids stored in:

```python
adata.obsm["spatial"]
```

were compared against ROI polygons using:

```python
matplotlib.path.Path.contains_point()
```

Cells whose centroids fell inside a polygon were assigned the corresponding islet identifier.

ROI annotations were stored within:

```python
adata.obs["Islets"]
```

allowing direct separation of endocrine and exocrine compartments.

---

# PART 3 — Extraction of Islet Cells

Cells assigned to manually annotated islets were extracted:

```python
my_filter = adata.obs["Islets"].notnull()
islet_cells = adata[my_filter]
```

These cells represent the endocrine compartment and include endocrine, vascular, stromal, and immune populations residing within islets.

Raw transcript counts were preserved in:

```python
islet_cells.layers["counts"]
```

for downstream differential expression analysis.

---

# PART 4 — Islet Cell Clustering and Annotation

The islet dataset was processed using a standard Scanpy workflow.

Highly variable genes were identified using the Seurat v3 method:

```python
sc.pp.highly_variable_genes()
```

followed by:

- Library-size normalization
- Log-transformation
- Principal component analysis (PCA)
- Neighborhood graph construction
- UMAP dimensionality reduction
- Leiden clustering

implemented using:

```python
sc.pp.normalize_total()
sc.pp.log1p()
sc.pp.pca()
sc.pp.neighbors()
sc.tl.umap()
sc.tl.leiden()
```

Cluster identities were assigned using canonical lineage markers visualized through differential expression analysis and dot plots.

Marker panels included:

| Cell Type | Representative Markers |
|------------|----------------------|
| Endothelial Cells | PECAM1, PLVAP, VWF, FLT1, SOX18, CLDN5 |
| Pericytes | PDGFRB, CSPG4, RGS5, ACE2 |
| Beta Cells | IAPP, PDX1, MAFA, UCN3 |
| Alpha Cells | GCG |
| Delta Cells | SST |
| PP Cells | PPY |
| Ductal Cells | CFTR, KRT19 |
| Immune Cells | CD8A, CD4, CD79A |

Cluster-specific marker genes were identified using:

```python
sc.tl.rank_genes_groups()
```

and visualized using:

```python
sc.pl.dotplot()
```

---

# PART 5 — Vascular Population Refinement

To better resolve endothelial and mural populations, the initial vascular cluster was isolated and re-clustered using Leiden clustering restricted to the vascular compartment:

```python
sc.tl.leiden(
    restrict_to=("clusters", ["5"])
)
```

Subclusters were annotated based on canonical marker expression.

Pericytes were identified by expression of:

- RGS5
- PDGFRB
- CSPG4
- ACE2

Endothelial cells were identified by expression of:

- PECAM1
- PLVAP
- VWF
- FLT1

Subclusters were manually merged and relabeled to generate final vascular annotations.

Final vascular labels were stored in:

```python
islet_cells.obs["leiden1"]
```

yielding:

| Final Label | Identity |
|-------------|----------|
| Pericytes | Mural cells |
| Endothelial Cells | Vascular endothelial cells |

The resulting vascular subsets:

```python
pericyteislet_filtered
```

and

```python
endoislet_filtered
```

represent islet-associated pericytes and endothelial cells respectively.

---

# PART 6 — Extraction of Exocrine Cells

To generate a matched exocrine reference population, all cells located outside manually annotated islets were extracted.

Cells were required to be absent from every ROI category:

```python
sub_cell_by_gene = adata[
    adata.obs[roi_category].isnull()
]
```

This population represents the surrounding exocrine pancreas and contains acinar, ductal, vascular, stromal, immune, and other non-endocrine cell types.

---

# PART 7 — Exocrine Cell Clustering and Annotation

The same processing workflow applied to islet cells was repeated for exocrine cells.

Steps included:

1. Quality control filtering
2. Highly variable gene selection
3. Library-size normalization
4. Log-transformation
5. PCA
6. Neighborhood graph construction
7. UMAP embedding
8. Leiden clustering
9. Marker-based annotation

using the same marker panels described for islet cells.

This generated exocrine endothelial and pericyte populations that could be directly compared against their endocrine-associated counterparts.

---

# Biological Objective

The purpose of this workflow is to generate spatially resolved endothelial and perivascular populations from human pancreatic MERSCOPE datasets while preserving anatomical context through manual islet annotation.

By integrating ROI-based endocrine segmentation with single-cell spatial transcriptomics, this pipeline enables:

- Identification of islet-associated endothelial cells
- Identification of islet-associated pericytes
- Comparison of endocrine and exocrine vascular niches
- Analysis of basement membrane transcriptional programs
- Analysis of extracellular matrix gene expression
- Characterization of vascular cell heterogeneity
- Investigation of endocrine niche organization

These annotated populations form the foundation for downstream analyses examining the cellular sources of basement membrane and extracellular matrix components within the human pancreatic vascular niche.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import squidpy as sq
import concord as ccd 
import os

sc.logging.print_header()

In [ ]:
import squidpy as sq
import scanpy as sc

# =====================================================
# ROI PATHS
# =====================================================
roi1_dir = "/Volumes/Samsung_4TB/HuP03A_S_data"
roi2_dir = "/Volumes/Samsung_4TB/HuP03A_L_data"

# =====================================================
# LOAD ROI 1
# =====================================================
adata1 = sq.read.vizgen(
    path=roi1_dir,
    counts_file="cell_by_gene.csv",
    meta_file="cell_metadata.csv",
    transformation_file="micron_to_mosaic_pixel_transform.csv",
)

# label ROI
adata1.obs["ROI"] = "ROI1"

# make cell IDs unique
adata1.obs_names = [
    f"ROI1_{x}"
    for x in adata1.obs_names
]

# =====================================================
# LOAD ROI 2
# =====================================================
adata2 = sq.read.vizgen(
    path=roi2_dir,
    counts_file="cell_by_gene.csv",
    meta_file="cell_metadata.csv",
    transformation_file="micron_to_mosaic_pixel_transform.csv",
)

# label ROI
adata2.obs["ROI"] = "ROI2"

# make cell IDs unique
adata2.obs_names = [
    f"ROI2_{x}"
    for x in adata2.obs_names
]

# =====================================================
# CONCATENATE
# =====================================================
adata = sc.concat(
    [adata1, adata2],
    join="outer",
    label="batch",
    keys=["ROI1", "ROI2"],
    index_unique=None
)

# =====================================================
# CHECK
# =====================================================
print(adata)

print(
    adata.obs["ROI"]
    .value_counts()
)

In [ ]:
vizgen_dir = '/Volumes/Samsung_4TB/HuP156_data'

adata = sq.read.vizgen(
    path=vizgen_dir,
    counts_file="cell_by_gene.csv",
    meta_file="cell_metadata.csv",
    transformation_file="micron_to_mosaic_pixel_transform.csv",
)

In [ ]:
adata

In [ ]:
adata.layers["counts"] = adata.X.copy()


In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=(50, 100, 200, 300), inplace=True)

In [ ]:
adata.obsm["blank_genes"].to_numpy().sum() / adata.var["total_counts"].sum() * 100

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(
    adata.obs["total_counts"],
    kde=False,
    ax=axs[0],
)

axs[1].set_title("Unique transcripts per cell")
sns.histplot(
    adata.obs["n_genes_by_counts"],
    kde=False,
    ax=axs[1],
)

axs[2].set_title("Transcripts per FOV")
sns.histplot(
    adata.obs.groupby("fov").sum()["total_counts"],
    kde=False,
    ax=axs[2],
)

axs[3].set_title("Volume of segmented cells")
sns.histplot(
    adata.obs["volume"],
    kde=False,
    ax=axs[3],
)

In [ ]:
sc.pp.filter_cells(adata, min_counts=10)


In [ ]:
adata

In [ ]:
adata.write("/Volumes/Samsung_4TB/Raw_fullandata_samples/ND/RHIP156.h5ad")

In [ ]:
adata.X

In [ ]:
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts")

In [ ]:
X_norm = sc.pp.normalize_total(adata, target_sum=1, inplace=False)['X']

In [ ]:
results_file1=('/Volumes/KM_2TB_SSD/HuP53A_data_new/basic53clustering3.h5ad')

In [ ]:
adata.write(results_file1)

In [ ]:
emp = adata.obs['clusters'].cat.categories

emp = ['Pericytes' if item == '4' else item for item in emp]
emp = ['Endothelial Cells' if item == '5' else item for item in emp]



adata.rename_categories('clusters', emp)

In [ ]:
# read in and inspect the ROI csv.
my_roi_csv_path = os.path.normpath(r'/Volumes/Samsung_4TB/HuP20A_data/HuP20-300G_ROI_07-08-24_17-53_geometry.csv')
my_rois_pd = pd.io.parsers.read_csv(my_roi_csv_path)
my_rois_pd

In [ ]:
# Create a new dataframe with several quality of life changes including:
# Split polygon type and its corresponding points
# Include the bounding box of the ROI for computational efficiency downstream

# Define new dataframe columns
new_roi_dict = {'ROI' : [],
                'geometry' : [],
                'points' : [],
                'dataset' : [],
                'group' : [],
                'x_low' : [],
                'x_high' : [],
                'y_low' : [],
                'y_high' : []
               }

# Iterate over old dataframe and copy / reformat the entries
for ind, row in my_rois_pd.iterrows():

    # Copy these exactly
    new_roi_dict['ROI'].append(row['ROI'])
    new_roi_dict['dataset'].append(row['dataset'])
    new_roi_dict['group'].append(row['group'])

    # Split out the geometry type
    geometry = row['geometry'].split(' ((')[0]
    new_roi_dict['geometry'].append(geometry)

    # Convert points from string into np.array
    # currently, this block assumes all are polygons
    points_string = row['geometry'].split('((')[1].split('))')[0]
    x_pos = []
    y_pos = []
    for pair in points_string.split(', '):
        x_pos.append(float(pair.split(' ')[0]))
        y_pos.append(float(pair.split(' ')[1]))
    points = np.array([x_pos, y_pos]).T
    new_roi_dict['points'].append(points)

    # Compute bounding box and add those columns
    new_roi_dict['x_low'].append(np.min(x_pos))
    new_roi_dict['x_high'].append(np.max(x_pos))
    new_roi_dict['y_low'].append(np.min(y_pos))
    new_roi_dict['y_high'].append(np.max(y_pos))

# Create new dataframe and inspect
roi_df = pd.DataFrame.from_dict(new_roi_dict)
roi_df

In [ ]:
# Define the new ROI categories

roi_categories = set(list(my_rois_pd['group']))
roi_categories

In [ ]:
import matplotlib.path as mpltPath

# Generate the new columns for the ROIs

# Initialize a dictionary to keep track of these new columns
new_adata_columns_dict = {}

for roi_category in roi_categories:
    new_adata_columns_dict[roi_category] = []

# For each cell, determine if it belongs to any ROI
for cell_positions in adata.obsm['spatial']:

    # Initially assume cell belongs to no ROI
    for roi_category in roi_categories:
        new_adata_columns_dict[roi_category].append(None)

    # Get the cell centroid
    x_pos = cell_positions[0]
    y_pos = cell_positions[1]

    # Filter ROIs to only include ones where the cell is in the bounding box
    candidate_rois = roi_df[(roi_df['x_low'] < x_pos) & (roi_df['x_high'] > x_pos) & (roi_df['y_low'] < y_pos) & (roi_df['y_high'] > y_pos)]

    # For each candidate ROI, find out exactly if the cell is inside
    for ind, candidate_roi in candidate_rois.iterrows():
        roi_path = mpltPath.Path(candidate_roi['points'], closed=True)
        in_path = roi_path.contains_point((x_pos, y_pos))
        if in_path:
            new_adata_columns_dict[candidate_roi['group']].pop() # Need to pop() bc assumed earlier cell did not belong to this ROI category
            new_adata_columns_dict[candidate_roi['group']].append(candidate_roi['ROI'])

In [ ]:
# Update the Anndata structure with the new obs
for roi_category in roi_categories:
    adata.obs[roi_category] = new_adata_columns_dict[roi_category]

In [ ]:
adata

In [ ]:
adata.obs['Islets'].value_counts()

In [ ]:
my_filter = adata.obs['Islets'].notnull()
islet_cells=adata[my_filter]
islet_cells

In [ ]:
islet_cells.layers["counts"] = islet_cells.X.copy()
sc.pp.highly_variable_genes(islet_cells, flavor="seurat_v3", n_top_genes=4000)
sc.pp.normalize_total(
    islet_cells, target_sum=1, exclude_highly_expressed=True,
    max_fraction=0.2, inplace=False
)['X']
sc.pp.log1p(islet_cells)
sc.pp.pca(islet_cells)
sc.pp.neighbors(islet_cells)
sc.tl.umap(islet_cells)
sc.tl.leiden(islet_cells, resolution=0.5, key_added='clusters')

In [ ]:
sc.pl.umap(islet_cells, color="clusters")
sc.pl.embedding(islet_cells, basis="spatial", color="clusters")

In [ ]:
sc.tl.rank_genes_groups(islet_cells, "clusters", method="wilcoxon")
sc.pl.rank_genes_groups(islet_cells, n_genes=25, sharey=False)

In [ ]:
marker_genes_dict = {
    "B-cell": ["CD79A", "CD19"],
    "Macrophages": ["CCR3", "CXCR3", "C1QC"],
    "Cadherins": ["CDH1", "CDH5"],
    "Beta Cells": ["IAPP", "GCK", "PDX1", "MAFA", 'UCN3', 'GLP1R', 'MAFB', 'NEUROD1', 'NKX6-1', 'PAX4', 'SOX1', 'SDC4'],
    "Pericytes": ["CSPG4", "PDGFRB", "S100A1", "SYP", 'ACE2', 'F3', 'RGS5', 'SFN', 'TMEM123'],
    "Endothelial": ['PLVAP', 'PECAM1', 'VWF', 'HSPG2', 'FLT1', 'RAMP2', 'PCAT19', 'CD34', 'SOX18', 'EGFL7', 'IFI27', 'ENG', 'PLPP1', 
                     'CLDN5', 'CLEC14A', 'CD93', 'MRTFB', 'TGFBR2', 'ESAM', 'KDR', 'TCIM', 'LIFR', 'SPARCL1', 'ITGA6', 'ICAM1'],
    "T-cell": ["CD8A", "CD4", "CD52", 'IL4', 'IL2', 'IL2RB', 'IL2RG'],
    "Ductal Cells": ["CFTR", "KRT19", "DCDC2", 'PROM1', 'SOX9', 'SPP1', 'TUBB3', 'VAT1L'], 
    "Endocrine Markers": ['CHGA', 'UCHL1', 'ISL1', 'CCDC186', 'SIMC1', 'NOS1', 'ERLIN1'], 
    "Alpha Cells": ['GCG', 'BEND4', 'IRX1', 'IRX2', 'LOXL4', 'RPL14'], 
    "Delta Cells": ['SST', 'SSTR2', 'HHEX', 'BMI1', 'VIP', 'RSPH1'], 
    "Epsilon Cells": ['GHR', 'LGALS3BP', 'POGK', 'SERGEF', 'VTN'], 
    "PPY Cells": ['PPY', 'PAX6', 'SERTM1'], 
    "Nerve Cells": ['CBX2', 'CEACAM1', 'GRIA2', 'MNX1', 'NCAM1', 'NPY', 'TRPV1']
}

In [ ]:
sc.pl.dotplot(islet_cells, marker_genes_dict, groupby='clusters')

In [ ]:
sc.tl.leiden(islet_cells, restrict_to=('clusters', ['5']), resolution=0.5, key_added='leiden1')

In [ ]:
sc.pl.dotplot(islet_cells, marker_genes_dict, groupby='leiden1')

In [ ]:
islet_cells.obs['leiden1'][islet_cells.obs['leiden1'].isin(['5,3', '5,5', '5,4'])]='5,0'
islet_cells.obs['leiden1'][islet_cells.obs['leiden1']=='5,2']='5,1'

islet_cells.obs['leiden1']=islet_cells.obs['leiden1'].astype('str').astype('category')
### Reorder and rename the Leiden
islet_cells.obs['leiden1'].cat.rename_categories(np.arange(len(np.unique(islet_cells.obs['leiden1']))).astype('str'))

In [ ]:
emp = islet_cells.obs['leiden1'].cat.categories

emp = ['Pericytes' if item == '5,1' else item for item in emp]
emp = ['Endothelial Cells' if item == '5,0' else item for item in emp]



islet_cells.rename_categories('leiden1', emp)

In [ ]:
sc.pl.umap(islet_cells, color='leiden1')

In [ ]:
pericyteislet_filtered = islet_cells[islet_cells.obs['leiden1'] == 'Pericytes']
pericyteislet_filtered

In [ ]:
endoislet_filtered = islet_cells[islet_cells.obs['leiden1'] == 'Endothelial Cells']
endoislet_filtered

In [ ]:
# Get all cells not in an ROI

sub_cell_by_gene = adata # remember this creates a view, not a full copy

# Iterate over all roi categories
for roi_category in roi_categories:
    sub_cell_by_gene = sub_cell_by_gene[sub_cell_by_gene.obs[roi_category].isnull()]

sub_cell_by_gene

In [ ]:
islet_cells

In [ ]:
sc.pp.calculate_qc_metrics(sub_cell_by_gene, percent_top=(50, 100, 200, 300), inplace=True)

In [ ]:
sub_cell_by_gene.obsm["blank_genes"].to_numpy().sum() / sub_cell_by_gene.var["total_counts"].sum() * 100

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(
    sub_cell_by_gene.obs["total_counts"],
    kde=False,
    ax=axs[0],
)

axs[1].set_title("Unique transcripts per cell")
sns.histplot(
    sub_cell_by_gene.obs["n_genes_by_counts"],
    kde=False,
    ax=axs[1],
)

In [ ]:
sc.pl.scatter(sub_cell_by_gene, x="total_counts", y="n_genes_by_counts")

In [ ]:
sc.pp.filter_cells(sub_cell_by_gene, min_counts=25)

In [ ]:
sub_cell_by_gene

In [ ]:
sub_cell_by_gene.layers["counts"] = sub_cell_by_gene.X.copy()
sc.pp.highly_variable_genes(sub_cell_by_gene, flavor="seurat_v3", n_top_genes=4000)
sc.pp.normalize_total(
    sub_cell_by_gene, target_sum=1, exclude_highly_expressed=True,
    max_fraction=0.2, inplace=False
)['X']
sc.pp.log1p(sub_cell_by_gene)
sc.pp.pca(sub_cell_by_gene)
sc.pp.neighbors(sub_cell_by_gene)
sc.tl.umap(sub_cell_by_gene)
sc.tl.leiden(sub_cell_by_gene, resolution=0.5, key_added='clusters')

In [ ]:
sc.pl.dotplot(sub_cell_by_gene, marker_genes_dict, groupby='clusters')